In [ ]:
# 1. Clona o repositório do GitHub
!git clone https://github.com/guidias-sketch/Algebra-linear-multiplicacao-de-matrizes-densas-GEMM.git

# 2. Entra na pasta do projeto
%cd Algebra-linear-multiplicacao-de-matrizes-densas-GEMM

# 3. Compila a biblioteca .so usando o Makefile
!make

In [ ]:
import numpy as np
import ctypes
import time

# Carrega a biblioteca compilada na célula anterior
lib = ctypes.CDLL('/content/Algebra-linear-multiplicacao-de-matrizes-densas-GEMM/libgemm.so')

# Define os tipos de argumentos que a função em C++ espera
lib.run_benchmarks.argtypes = [
    ctypes.c_int,
    ctypes.POINTER(ctypes.c_float), # h_A
    ctypes.POINTER(ctypes.c_float), # h_B
    ctypes.POINTER(ctypes.c_float), # h_C_cpu
    ctypes.POINTER(ctypes.c_float), # h_C_naive
    ctypes.POINTER(ctypes.c_float)  # h_C_tiled
]

def benchmark_gemm(N=512): 
    print(f"Iniciando benchmarks para matriz {N}x{N}...\n")
    A = np.random.rand(N, N).astype(np.float32)
    B = np.random.rand(N, N).astype(np.float32)
    
    C_cpu = np.zeros((N, N), dtype=np.float32)
    C_naive = np.zeros((N, N), dtype=np.float32)
    C_tiled = np.zeros((N, N), dtype=np.float32)

    # Referência NumPy
    start = time.time()
    C_numpy = np.matmul(A, B)
    t_numpy = time.time() - start
    print(f"Tempo NumPy (cuBLAS/MKL): {t_numpy:.4f}s")

    # Preparando Ponteiros para passar ao C++
    ptr_A = A.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    ptr_B = B.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    ptr_C_cpu = C_cpu.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    ptr_C_naive = C_naive.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    ptr_C_tiled = C_tiled.ctypes.data_as(ctypes.POINTER(ctypes.c_float))

    # Execução unificada em C++/CUDA
    start = time.time()
    lib.run_benchmarks(N, ptr_A, ptr_B, ptr_C_cpu, ptr_C_naive, ptr_C_tiled)
    t_total = time.time() - start
    print(f"Tempo total do backend compilado (CPU + Kernels): {t_total:.4f}s")

    # Validação dos resultados para garantir que a GPU não gerou lixo
    np.testing.assert_allclose(C_numpy, C_cpu, atol=1e-2)
    np.testing.assert_allclose(C_numpy, C_naive, atol=1e-2)
    np.testing.assert_allclose(C_numpy, C_tiled, atol=1e-2)
    print("\n✅ Sucesso! Todas as implementações coincidem com o NumPy.")

# Executa o teste
benchmark_gemm(512)